# PartPilot — End-to-End Colab Demo

Upload a car-part photo → remove background → classify category (EfficientNetB0) → similarity search in that category's FAISS index → top matched SKU + score.

**Run the cells top to bottom.** Steps:
1. Install libraries
2. Get the project code (clone the `demo` branch)
3. Upload your images
4. Convert images to JPG
5. Train Brain 1 (EfficientNetB0) + save `labels.json`
6. Build per-category FAISS indexes
7. Test: upload a photo and get the matching SKU

> Note: this is a small demo dataset (2 categories), so accuracy is only illustrative — it proves the full flow works.

## 1. Install libraries

In [ ]:
!pip -q install rembg onnxruntime open_clip_torch faiss-cpu tensorflow pillow

## 2. Get the project code (the `demo` branch)

In [ ]:
import os, sys
%cd /content
!rm -rf Parts-Detection-Hackathon
!git clone -q https://github.com/zubariyasulekaz/Parts-Detection-Hackathon.git
%cd Parts-Detection-Hackathon
!git checkout -q demo
%cd partpilot
PARTPILOT = '/content/Parts-Detection-Hackathon/partpilot'
sys.path.insert(0, PARTPILOT)
print('Working dir:', os.getcwd())

## 3. Upload your images

The images are **not** in the repo (they're git-ignored). On your laptop, zip the
`partpilot/datasets/images` folder into `images.zip` so the zip contains
`images/<SKU>/...`, then upload it below.

*(Alternatively, mount Google Drive and copy the folder in.)*

In [ ]:
import zipfile, os
from google.colab import files

uploaded = files.upload()  # choose images.zip
zip_name = list(uploaded.keys())[0]
os.makedirs('datasets', exist_ok=True)
with zipfile.ZipFile(zip_name) as z:
    z.extractall('datasets')   # -> datasets/images/<SKU>/...

skus = sorted(os.listdir('datasets/images'))
print(f'{len(skus)} SKU folders:', skus)

## 4. Convert images to JPG

Turns `.avif` / `.webp` into `.jpg` (which the models can read) and deletes the originals.

In [ ]:
!python scripts/convert_images_to_jpg.py --delete

## 5. Train Brain 1 (EfficientNetB0) + save `labels.json`

Groups the per-SKU folders into category folders (using `catalog.csv`), trains a
classifier, and saves the model **and** `labels.json` into
`backend/models/classifier/`. The label names are the catalog categories, so they
line up with the FAISS index filenames automatically.

In [ ]:
import csv, json, shutil, pathlib
import tensorflow as tf
from tensorflow import keras

ROOT = pathlib.Path.cwd()  # .../partpilot

# --- group SKU images into category folders using catalog.csv ---
sku_category = {}
with open('datasets/catalog.csv', newline='', encoding='utf-8-sig') as f:
    for row in csv.DictReader(f):
        sku_category[row['sku']] = row['category'].strip()

TRAIN_DIR = pathlib.Path('/content/brain1_dataset')
if TRAIN_DIR.exists():
    shutil.rmtree(TRAIN_DIR)
for sku, category in sku_category.items():
    src = ROOT / 'datasets' / 'images' / sku
    if not src.is_dir():
        continue
    dst = TRAIN_DIR / category
    dst.mkdir(parents=True, exist_ok=True)
    for img in src.glob('*'):
        if img.suffix.lower() in {'.jpg', '.jpeg', '.png'}:
            shutil.copy(img, dst / f'{sku}_{img.name}')
print('Images per category:', {p.name: len(list(p.glob('*'))) for p in TRAIN_DIR.iterdir()})

# --- datasets ---
IMG_SIZE = (224, 224)
BATCH = 8
train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR, validation_split=0.2, subset='training', seed=42,
    image_size=IMG_SIZE, batch_size=BATCH)
val_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR, validation_split=0.2, subset='validation', seed=42,
    image_size=IMG_SIZE, batch_size=BATCH)
class_names = train_ds.class_names
print('Classes (output order):', class_names)

# --- model (EfficientNetB0 transfer learning; preprocess baked into graph) ---
aug = keras.Sequential([
    keras.layers.RandomFlip('horizontal'),
    keras.layers.RandomRotation(0.1),
    keras.layers.RandomZoom(0.1),
])
base = tf.keras.applications.EfficientNetB0(
    include_top=False, weights='imagenet', input_shape=(224, 224, 3))
base.trainable = False

inp = keras.Input((224, 224, 3))
x = aug(inp)
x = tf.keras.applications.efficientnet.preprocess_input(x)
x = base(x, training=False)
x = keras.layers.GlobalAveragePooling2D()(x)
x = keras.layers.Dropout(0.2)(x)
out = keras.layers.Dense(len(class_names), activation='softmax')(x)
model = keras.Model(inp, out)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(train_ds, validation_data=val_ds, epochs=15)

# --- save model + labels sidecar where the backend expects them ---
out_dir = ROOT / 'backend' / 'models' / 'classifier'
out_dir.mkdir(parents=True, exist_ok=True)
model.save(out_dir / 'brain1_classifier.keras')
(out_dir / 'labels.json').write_text(json.dumps(class_names))
print('Saved brain1_classifier.keras + labels.json ->', out_dir)

## 6. Build per-category FAISS indexes

One index per category (`brake_pads.faiss`, `oil_filter.faiss`). `--remove-bg`
cleans each catalog image the same way the runtime cleans an uploaded query,
so the fingerprints match.

In [ ]:
!python scripts/build_faiss_indexes.py --remove-bg
import os
print('Indexes:', os.listdir('backend/models/faiss'))

## 7. Test it — upload a photo, get the matching SKU

Runs the real pipeline via the backend orchestrator: rembg → classify → search → catalog lookup.

In [ ]:
import io
from PIL import Image
from google.colab import files
from backend.api.dependencies import get_orchestrator

uploaded = files.upload()  # choose a test part photo
name = list(uploaded.keys())[0]
img = Image.open(io.BytesIO(uploaded[name])).convert('RGB')

orchestrator = get_orchestrator()
result = orchestrator.run(img, top_k=5)
pred = result.prediction

print(f'Predicted category : {pred.predicted_category}  (confidence {pred.confidence:.1%})')
print(f'Search time        : {pred.search_time_ms:.0f} ms')
print('Top matches:')
for r in pred.results:
    print(f'   {r.sku:10}  similarity {r.similarity_score:.3f}')
if result.product:
    print(f'\nBest match product : {result.product.product_name}  (SKU {result.product.sku})')
if result.recommendation and result.recommendation.alternatives:
    print('Alternatives       :', [p.sku for p in result.recommendation.alternatives])